# Step 3 — Synthetic Drift Injection

**Thesis:** Drift-Aware Selective Updating of Two-Stage Tabular ML Pipelines  
**Goal:** Build a `DriftInjector` class and visualize the effect of each drift type and severity level.

Supported drift modes:
- **Covariate shift** — shifts numeric feature distributions by `severity × std`
- **Concept drift** — randomly flips labels with probability proportional to severity
- **Both** — applies both transformations simultaneously

Reference implementation: `drift_framework/drift/injector.py`

## 3.1 Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from drift_framework.data.loader import load_dataset

SEED = 42
bundle = load_dataset("adult")
print("Data loaded — X_post shape:", bundle.X_post.shape)

## 3.2 Severity multipliers

In [ ]:
# Covariate shift: how many standard deviations to shift the features
SEVERITY_COVARIATE = {"low": 0.5, "medium": 1.5, "high": 3.0}

# Concept drift: probability of flipping each label (capped at 45%)
SEVERITY_CONCEPT = {"low": 0.08, "medium": 0.25, "high": 0.45}

print("Covariate multipliers:", SEVERITY_COVARIATE)
print("Concept flip probs:   ", SEVERITY_CONCEPT)

## 3.3 DriftInjector class

In [ ]:
from typing import Literal, Optional, Tuple


class DriftInjector:
    """
    Injects synthetic drift into a dataset split.

    Parameters
    ----------
    drift_type : "covariate" | "concept" | "both"
    severity   : "low" | "medium" | "high"
    start_idx  : row index at which drift begins
    duration   : None = permanent; int = temporary spike of that many rows
    random_state : RNG seed
    """

    def __init__(
        self,
        drift_type: Literal["covariate", "concept", "both"],
        severity: Literal["low", "medium", "high"],
        start_idx: int = 0,
        duration: Optional[int] = None,
        random_state: int = SEED,
    ):
        if drift_type not in ("covariate", "concept", "both"):
            raise ValueError(f"drift_type must be 'covariate', 'concept', or 'both'; got '{drift_type}'")
        if severity not in SEVERITY_COVARIATE:
            raise ValueError(f"severity must be 'low', 'medium', or 'high'; got '{severity}'")

        self.drift_type = drift_type
        self.severity = severity
        self.start_idx = start_idx
        self.duration = duration
        self.random_state = random_state

        self._covariate_multiplier = SEVERITY_COVARIATE[severity]
        self._flip_prob = SEVERITY_CONCEPT[severity]
        self._num_features: list = []
        self._feature_stds: Optional[dict] = None

    def fit(self, X_ref: pd.DataFrame, num_features: list) -> "DriftInjector":
        """Compute per-feature stds from the reference split."""
        self._num_features = [c for c in num_features if c in X_ref.columns]
        self._feature_stds = {col: float(X_ref[col].std()) for col in self._num_features}
        return self

    def inject(self, X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, pd.Series]:
        """Apply drift to rows [start_idx : start_idx + duration]."""
        if self._feature_stds is None:
            raise RuntimeError("Call fit() with reference data before inject().")

        rng = np.random.default_rng(self.random_state)
        X_out = X.copy()
        y_out = y.copy()

        end_idx = (
            min(self.start_idx + self.duration, len(X))
            if self.duration is not None
            else len(X)
        )
        affected = np.zeros(len(X), dtype=bool)
        affected[self.start_idx:end_idx] = True

        if self.drift_type in ("covariate", "both"):
            for col in self._num_features:
                shift = self._covariate_multiplier * self._feature_stds[col]
                X_out.loc[affected, col] = X_out.loc[affected, col] + shift

        if self.drift_type in ("concept", "both"):
            n_affected = int(affected.sum())
            flip_flags = rng.random(n_affected) < self._flip_prob
            flip_idxs = np.where(affected)[0][flip_flags]
            y_arr = y_out.values.copy()
            y_arr[flip_idxs] = 1 - y_arr[flip_idxs]
            y_out = pd.Series(y_arr, index=y_out.index, name=y_out.name)

        return X_out, y_out

## 3.4 Covariate shift — feature distribution shift

In [ ]:
# Pick 3 numeric features to visualize
VIZ_FEATURES = bundle.num_features[:3]
print("Visualizing features:", VIZ_FEATURES)

injector_cov = DriftInjector(drift_type="covariate", severity="high", start_idx=0)
injector_cov.fit(bundle.X_ref, bundle.num_features)
X_cov, _ = injector_cov.inject(bundle.X_post, bundle.y_post)

fig, axes = plt.subplots(1, len(VIZ_FEATURES), figsize=(14, 4))
for ax, col in zip(axes, VIZ_FEATURES):
    ax.hist(bundle.X_post[col].dropna(), bins=40, alpha=0.6, label="original", color="steelblue")
    ax.hist(X_cov[col].dropna(), bins=40, alpha=0.6, label="drifted (high)", color="tomato")
    ax.set_title(col)
    ax.legend()
plt.suptitle("Covariate Shift — high severity", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Compare mean shift across all severity levels for the first feature
col = bundle.num_features[0]
orig_mean = bundle.X_post[col].mean()
print(f"Feature: {col}  (original mean: {orig_mean:.4f})")

for sev in ["low", "medium", "high"]:
    inj = DriftInjector(drift_type="covariate", severity=sev)
    inj.fit(bundle.X_ref, bundle.num_features)
    X_d, _ = inj.inject(bundle.X_post, bundle.y_post)
    drift_mean = X_d[col].mean()
    print(f"  {sev:8s} drift mean: {drift_mean:.4f}  (shift = {drift_mean - orig_mean:+.4f})")

## 3.5 Concept drift — label flipping

In [ ]:
orig_rate = bundle.y_post.mean()
print(f"Original positive label rate: {orig_rate:.4f}")

for sev in ["low", "medium", "high"]:
    inj = DriftInjector(drift_type="concept", severity=sev)
    inj.fit(bundle.X_ref, bundle.num_features)
    _, y_d = inj.inject(bundle.X_post, bundle.y_post)
    drifted_rate = y_d.mean()
    n_flipped = (y_d != bundle.y_post).sum()
    print(f"  {sev:8s} — positive rate: {drifted_rate:.4f}  |  labels flipped: {n_flipped}/{len(y_d)}")

## 3.6 Both drift types simultaneously

In [ ]:
injector_both = DriftInjector(drift_type="both", severity="high", start_idx=0)
injector_both.fit(bundle.X_ref, bundle.num_features)
X_both, y_both = injector_both.inject(bundle.X_post, bundle.y_post)

col = bundle.num_features[0]
print(f"Feature '{col}':")
print(f"  Original mean : {bundle.X_post[col].mean():.4f}")
print(f"  Drifted  mean : {X_both[col].mean():.4f}")
print(f"Original label rate : {bundle.y_post.mean():.4f}")
print(f"Drifted  label rate : {y_both.mean():.4f}")

## 3.7 Temporal control — `start_idx` and `duration`

In [ ]:
n_rows = len(bundle.X_post)
mid = n_rows // 2

# Drift starts at the midpoint, lasts for 200 rows (temporary spike)
inj_temp = DriftInjector(drift_type="covariate", severity="high", start_idx=mid, duration=200)
inj_temp.fit(bundle.X_ref, bundle.num_features)
X_temp, _ = inj_temp.inject(bundle.X_post, bundle.y_post)

col = bundle.num_features[0]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(bundle.X_post[col].values, alpha=0.6, label="original", color="steelblue")
ax.plot(X_temp[col].values, alpha=0.6, label="drifted (temp spike)", color="tomato")
ax.axvline(mid, color="gray", linestyle="--", label=f"start_idx={mid}")
ax.axvline(mid + 200, color="gray", linestyle=":", label=f"end_idx={mid+200}")
ax.set_title(f"Temporary covariate shift — feature: {col}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Rows affected: {mid} to {mid+200} (out of {n_rows})")

## 3.8 Sanity checks

In [ ]:
# 1. Original DataFrames must be unchanged (injector works on copies)
inj = DriftInjector(drift_type="both", severity="high")
inj.fit(bundle.X_ref, bundle.num_features)
X_orig_before = bundle.X_post.copy()
y_orig_before = bundle.y_post.copy()
X_d, y_d = inj.inject(bundle.X_post, bundle.y_post)

pd.testing.assert_frame_equal(bundle.X_post, X_orig_before)
pd.testing.assert_series_equal(bundle.y_post, y_orig_before)
print("Original data unchanged after inject() — OK")

# 2. Covariate shift actually changes numeric columns
inj_cov = DriftInjector(drift_type="covariate", severity="high")
inj_cov.fit(bundle.X_ref, bundle.num_features)
X_cov, _ = inj_cov.inject(bundle.X_post, bundle.y_post)
col = bundle.num_features[0]
assert X_cov[col].mean() != bundle.X_post[col].mean(), "Covariate shift had no effect"
print("Covariate shift changes feature values — OK")

# 3. Concept drift changes labels
inj_con = DriftInjector(drift_type="concept", severity="high")
inj_con.fit(bundle.X_ref, bundle.num_features)
_, y_con = inj_con.inject(bundle.X_post, bundle.y_post)
n_flipped = (y_con != bundle.y_post).sum()
assert n_flipped > 0, "Concept drift flipped 0 labels"
print(f"Concept drift flipped {n_flipped} labels — OK")

# 4. Temporary drift only affects the specified window
inj_t = DriftInjector(drift_type="covariate", severity="high", start_idx=100, duration=50)
inj_t.fit(bundle.X_ref, bundle.num_features)
X_t, _ = inj_t.inject(bundle.X_post, bundle.y_post)
# Before start_idx: unchanged
pd.testing.assert_frame_equal(X_t.iloc[:100], bundle.X_post.iloc[:100])
# After end_idx: unchanged
pd.testing.assert_frame_equal(X_t.iloc[150:], bundle.X_post.iloc[150:])
print("Temporary drift only affects specified window — OK")

# 5. Compare with framework injector
from drift_framework.drift.injector import DriftInjector as FWInjector
fw_inj = FWInjector(drift_type="covariate", severity="high")
fw_inj.fit(bundle.X_ref, bundle.num_features)
X_fw, _ = fw_inj.inject(bundle.X_post, bundle.y_post)
pd.testing.assert_frame_equal(X_cov, X_fw)
print("Notebook injector matches framework injector — OK")

print("\nAll sanity checks passed!")